# SeoulMate Structured Query Parser + RAG/Weather MCP LangGraph

사용자 질문을 한 번의 Structured Output으로 분석하고, 같은 결과에서 Intent·Domain Task·데이터 소스 사용 여부를 결정합니다.

```text
START
  ↓
Structured Query Parser
  ├─ general_response → 일반 LLM → END
  ├─ RAG_ONLY → Domain Agent → RAG → RAG 전용 최종 GPT
  ├─ RAG_MCP  → Domain Agent → RAG + Weather MCP → 재랭킹 → 결합 최종 GPT
  └─ MCP_ONLY → Weather MCP → MCP 전용 최종 GPT
```

핵심 원칙:

- 별도의 소스 라우터 GPT를 추가하지 않습니다.
- 기존 Structured Query Parser가 `source_mode`까지 함께 반환합니다.
- 위치 좌표는 LLM이 만들지 않고 `user_context`에서 MCP 요청에 결합합니다.
- 이 노트북은 Domain Agent 요청과 최종 GPT 입력 payload까지 만듭니다. 실제 VectorDB 검색은 각 도메인 Agent 구현에 연결합니다.

## 1. 패키지 설치

처음 실행하는 환경에서만 아래 셀의 주석을 해제합니다. Weather MCP 실호출은 SeoulMate 백엔드의 `.venv`와 `services/weather_mcp_client.py`를 사용합니다.

In [1]:
# %pip install -U \
#     "pydantic>=2.7" \
#     "langchain>=1.0" \
#     "langgraph>=1.0" \
#     "langchain-openai>=1.0" \
#     "python-dotenv>=1.0"

## 2. 기본 설정과 import

In [2]:
import json
import os
import sys
from dataclasses import dataclass
from datetime import date
from enum import Enum
from pathlib import Path
from typing import Any, Literal, Optional, Protocol

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator
from typing_extensions import TypedDict

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

BACKEND_DIR = Path(
    os.getenv(
        "SEOULMATE_BACKEND_DIR",
        r"C:\Users\user\Desktop\seoulmate\SeoulMate\backend",
    )
).resolve()

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

load_dotenv(BACKEND_DIR / ".env")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))
WEATHER_MCP_URL = os.getenv("WEATHER_MCP_URL", "http://127.0.0.1:8001/mcp")

print("백엔드 절대 경로:", BACKEND_DIR)
print("사용 모델:", MODEL_NAME)
print("OpenAI API 사용 가능:", HAS_OPENAI_KEY)
print("Weather MCP:", WEATHER_MCP_URL)

백엔드 절대 경로: C:\Users\user\Desktop\seoulmate\SeoulMate\backend
사용 모델: gpt-4o-mini
OpenAI API 사용 가능: True
Weather MCP: http://127.0.0.1:8001/mcp


## 3. Structured Query 스키마

`source_mode`은 일반 대화를 제외한 데이터 사용 경로입니다.

| 값 | 의미 |
|---|---|
| `rag_only` | 도메인 RAG만 사용 |
| `rag_mcp` | 도메인 RAG와 Weather MCP를 함께 사용 |
| `mcp_only` | 장소 검색 없이 Weather MCP만 사용 |

날씨만 묻는 질문을 `general_response`로 섞지 않도록 `weather_information` Intent를 별도로 둡니다.

In [3]:
class Language(str, Enum):
    KO = "ko"
    EN = "en"
    JA = "ja"
    ZH = "zh"
    OTHER = "other"


class Intent(str, Enum):
    MULTI_DAY_ROUTE = "multi_day_route"
    DAY_TRIP_ROUTE = "day_trip_route"
    SINGLE_PLACE_RECOMMENDATION = "single_place_recommendation"
    WEATHER_INFORMATION = "weather_information"
    GENERAL_RESPONSE = "general_response"


class SourceMode(str, Enum):
    RAG_ONLY = "rag_only"
    RAG_MCP = "rag_mcp"
    MCP_ONLY = "mcp_only"


class Domain(str, Enum):
    CAFE = "cafe"
    RESTAURANT = "restaurant"
    ACCOMMODATION = "accommodation"
    ATTRACTION = "attraction"
    ETC = "etc"


class GeoPoint(BaseModel):
    latitude: float = Field(ge=-90, le=90)
    longitude: float = Field(ge=-180, le=180)


class UserContext(BaseModel):
    current_location: Optional[GeoPoint] = None
    location_name: Optional[str] = None
    timezone: str = "Asia/Seoul"


class SearchFilters(BaseModel):
    location: Optional[str] = None
    radius_km: Optional[float] = Field(default=None, gt=0, le=100)
    is_active: Optional[bool] = None
    start_date: Optional[date] = None
    end_date: Optional[date] = None
    time_window: Optional[str] = None
    party_size: Optional[int] = Field(default=None, ge=1, le=100)
    budget_min_krw: Optional[int] = Field(default=None, ge=0)
    budget_max_krw: Optional[int] = Field(default=None, ge=0)
    transportation: list[str] = Field(default_factory=list)
    accessibility: list[str] = Field(default_factory=list)
    required_features: list[str] = Field(default_factory=list)
    excluded_features: list[str] = Field(default_factory=list)

    @field_validator("transportation", "accessibility", "required_features", "excluded_features", mode="before")
    @classmethod
    def normalize_optional_lists(cls, value):
        return [] if value is None else value

    @model_validator(mode="after")
    def validate_ranges(self):
        if self.start_date and self.end_date and self.start_date > self.end_date:
            raise ValueError("start_date는 end_date보다 늦을 수 없습니다.")
        if (
            self.budget_min_krw is not None
            and self.budget_max_krw is not None
            and self.budget_min_krw > self.budget_max_krw
        ):
            raise ValueError("budget_min_krw는 budget_max_krw보다 클 수 없습니다.")
        return self


class TripPeriod(BaseModel):
    start_date: date
    end_date: date
    nights: int = Field(ge=0, le=6)
    days: int = Field(ge=1, le=7)

    @model_validator(mode="after")
    def validate_period(self):
        actual_days = (self.end_date - self.start_date).days + 1
        if actual_days != self.days or self.nights != self.days - 1:
            raise ValueError("날짜 범위, nights, days가 일치해야 합니다.")
        return self


class RouteBudget(BaseModel):
    total_krw: Optional[int] = Field(default=None, ge=0)
    daily_krw: Optional[int] = Field(default=None, ge=0)
    accommodation_total_krw: Optional[int] = Field(default=None, ge=0)
    meal_per_person_krw: Optional[int] = Field(default=None, ge=0)


class RouteRequest(BaseModel):
    destination: str = Field(min_length=1)
    period: TripPeriod
    adults: int = Field(default=1, ge=1, le=100)
    children: int = Field(default=0, ge=0, le=100)
    arrival_at: Optional[str] = None
    arrival_location: Optional[str] = None
    departure_at: Optional[str] = None
    departure_location: Optional[str] = None
    accommodation_id: Optional[str] = None
    preferred_accommodation_areas: list[str] = Field(default_factory=list)
    pace: Literal["relaxed", "normal", "packed"] = "normal"
    max_places_per_day: int = Field(default=5, ge=1, le=5)
    transportation: list[str] = Field(default_factory=list)
    budget: RouteBudget = Field(default_factory=RouteBudget)
    preferred_areas: list[str] = Field(default_factory=list)
    preferred_themes: list[str] = Field(default_factory=list)
    required_features: list[str] = Field(default_factory=list)
    excluded_features: list[str] = Field(default_factory=list)
    must_visit: list[str] = Field(default_factory=list)
    avoid_places: list[str] = Field(default_factory=list)


class QueryTask(BaseModel):
    task_id: str = Field(min_length=1)
    domain: Domain
    search_query: str = Field(min_length=1)
    themes: list[str] = Field(default_factory=list)
    desired_count: int = Field(default=1, ge=1, le=20)
    notes: Optional[str] = None
    # 루트 요청에서는 Task 하나가 방문 슬롯 하나를 의미한다.
    slot_id: Optional[str] = None
    day_number: Optional[int] = Field(default=None, ge=1, le=7)
    visit_date: Optional[date] = None
    start_time: Optional[str] = None
    end_date: Optional[date] = None
    end_time: Optional[str] = None
    filters: Optional[SearchFilters] = None

    @field_validator("themes", mode="before")
    @classmethod
    def remove_duplicate_themes(cls, values: list[str]) -> list[str]:
        values = values or []
        return list(dict.fromkeys(value.strip() for value in values if value.strip()))


class WeatherRequest(BaseModel):
    # LLM이 해석하는 날씨 요청. 좌표는 후속 노드가 UserContext에서 결합한다.

    query: str = Field(min_length=1)
    location_name: Optional[str] = None
    target_date: Optional[date] = None
    target_time: Optional[str] = None
    language: Language


class TravelQuery(BaseModel):
    model_config = ConfigDict(extra="forbid")

    language: Language
    intent: Intent
    source_mode: Optional[SourceMode] = None
    original_question: str = Field(min_length=1)
    normalized_question: str = Field(min_length=1)
    # 일반 추천은 validator에서 5개로 제한하고 루트만 최대 7일 x 5슬롯을 허용한다.
    tasks: list[QueryTask] = Field(default_factory=list, max_length=35)
    filters: SearchFilters = Field(default_factory=SearchFilters)
    weather_request: Optional[WeatherRequest] = None
    route_request: Optional[RouteRequest] = None
    general_response_instruction: Optional[str] = None

    @field_validator("filters", mode="before")
    @classmethod
    def normalize_optional_filters(cls, value):
        return {} if value is None else value

    @model_validator(mode="after")
    def validate_execution_contract(self):
        route_intents = {Intent.DAY_TRIP_ROUTE, Intent.MULTI_DAY_ROUTE, Intent.MODIFY_ROUTE}
        if self.intent not in route_intents and len(self.tasks) > 5:
            raise ValueError("일반 추천 Task는 최대 5개입니다.")
        if self.intent in route_intents:
            counts = {}
            for task in self.tasks:
                day = task.day_number or 1
                counts[day] = counts.get(day, 0) + 1
                if counts[day] > 5:
                    raise ValueError("루트의 하루 방문 슬롯은 최대 5개입니다.")
        if self.intent in {Intent.DAY_TRIP_ROUTE, Intent.MULTI_DAY_ROUTE}:
            if not self.route_request:
                raise ValueError("신규 루트 생성에는 route_request가 필요합니다.")
            if self.intent == Intent.DAY_TRIP_ROUTE and self.route_request.period.days != 1:
                raise ValueError("day_trip_route의 기간은 1일이어야 합니다.")
            if self.intent == Intent.MULTI_DAY_ROUTE and self.route_request.period.days < 2:
                raise ValueError("multi_day_route의 기간은 2일 이상이어야 합니다.")
            for task in self.tasks:
                if task.day_number and task.day_number > self.route_request.period.days:
                    raise ValueError("Task day_number가 여행 기간을 벗어났습니다.")
        if self.intent == Intent.GENERAL_RESPONSE:
            if self.source_mode is not None:
                raise ValueError("general_response에서는 source_mode가 없어야 합니다.")
            if self.tasks or self.weather_request:
                raise ValueError("general_response에서는 Task와 WeatherRequest가 없어야 합니다.")
            if not self.general_response_instruction:
                raise ValueError("general_response_instruction이 필요합니다.")
            return self

        if self.intent == Intent.WEATHER_INFORMATION:
            if self.source_mode != SourceMode.MCP_ONLY:
                raise ValueError("weather_information은 MCP_ONLY여야 합니다.")
            if self.tasks:
                raise ValueError("MCP_ONLY에서는 도메인 Task가 없어야 합니다.")
            if not self.weather_request:
                raise ValueError("MCP_ONLY에서는 weather_request가 필요합니다.")
            return self

        if self.source_mode not in {SourceMode.RAG_ONLY, SourceMode.RAG_MCP}:
            raise ValueError("여행 검색 Intent는 RAG_ONLY 또는 RAG_MCP여야 합니다.")
        if not self.tasks:
            raise ValueError("RAG를 사용하는 모드에서는 Task가 한 개 이상 필요합니다.")
        if self.source_mode == SourceMode.RAG_MCP and not self.weather_request:
            raise ValueError("RAG_MCP에서는 weather_request가 필요합니다.")
        if self.source_mode == SourceMode.RAG_ONLY and self.weather_request:
            raise ValueError("RAG_ONLY에서는 weather_request를 생성하지 않습니다.")
        return self

## 4. 후속 실행 모델과 LangGraph 상태

In [4]:
class SearchRequest(BaseModel):
    task_id: str
    domain: Domain
    query_text: str
    metadata_filters: dict[str, Any] = Field(default_factory=dict)
    top_k: int = Field(default=5, ge=1, le=50)


class AgentRequest(BaseModel):
    agent_name: str
    task_id: str
    domain: Domain
    search_request: SearchRequest


class WeatherMcpRequest(BaseModel):
    query: str
    lat: float
    lng: float
    language: str
    place_name: Optional[str] = None


class StructuredQueryParser(Protocol):
    async def ainvoke(self, inputs: dict[str, Any]) -> TravelQuery: ...


class GeneralAnswerGenerator(Protocol):
    async def ainvoke(self, inputs: dict[str, Any]) -> str: ...


class WeatherContextProvider(Protocol):
    async def ainvoke(self, inputs: dict[str, Any]) -> dict[str, Any]: ...


@dataclass
class GraphServices:
    query_parser: StructuredQueryParser
    general_answer_generator: GeneralAnswerGenerator
    weather_provider: WeatherContextProvider


class GraphState(TypedDict, total=False):
    question: str
    user_context: dict[str, Any]
    parsed_query: TravelQuery
    source_mode: str
    search_requests: list[SearchRequest]
    agent_requests: list[AgentRequest]
    weather_mcp_request: WeatherMcpRequest
    weather_context: dict[str, Any]
    final_answer_input: dict[str, Any]
    final_answer: str
    route_name: str
    error: str

## 5. Structured Query Parser 프롬프트

In [5]:
PARSER_SYSTEM_PROMPT = """
당신은 SeoulMate의 Structured Query Parser이자 데이터 소스 라우터입니다.
답변을 작성하지 말고 TravelQuery만 반환합니다.

[Intent]
- multi_day_route: 1박 이상 일정
- day_trip_route: 숙박 없는 하루 일정
- single_place_recommendation: 하나 또는 소수의 장소 추천
- weather_information: 장소 추천 없이 현재/미래 날씨만 질문
- general_response: 여행 처리와 날씨 조회가 모두 필요 없는 일반 질문

[SourceMode]
- rag_only: 장소·숙박·카페·식당·관광지 검색이 필요하지만 실제 현재/미래 날씨는 필요 없음
- rag_mcp: 도메인 검색과 실제 날씨 조회가 모두 필요함
- mcp_only: 장소 검색 없이 날씨 정보만 필요함
- general_response에서는 source_mode를 null로 반환

[판정 예시]
- "조용한 감성 카페 추천" → single_place_recommendation + rag_only
- "내일 저녁 조용한 감성 카페 추천" → single_place_recommendation + rag_mcp
- "내일 서울 날씨" → weather_information + mcp_only
- "비 오는 날 갈 문화시설 추천" → single_place_recommendation + rag_mcp
- "파이썬 리스트와 튜플 차이" → general_response + source_mode null

[Task]
- 하나의 Task는 하나의 domain만 가집니다.
- domain은 cafe, restaurant, accommodation, attraction, etc 중 하나입니다.
- 일반 추천 Task는 최대 5개입니다. day_trip_route/multi_day_route만 하루 최대 5개, 전체 최대 35개 슬롯 Task를 허용합니다.
- 루트 Task는 slot_id, day_number, visit_date, start_time을 반드시 채우고 Task 하나를 방문 슬롯 하나로 만듭니다.
- 당일 루트는 모든 Task의 day_number를 1로, 다일 루트는 날짜와 day_number가 일치하도록 만듭니다.
- 신규 당일·다일 루트에는 route_request를 반드시 만듭니다. destination과 period(start_date/end_date/nights/days)는 필수입니다.
- 인원, 도착·출발, 이동수단, pace, 예산, 선호 지역·테마, 필수·제외 조건은 사용자가 말한 값만 route_request에 넣습니다.
- day_trip_route는 nights=0/days=1, multi_day_route는 days>=2이고 nights=days-1이어야 합니다.
- 복합 질문은 독립적으로 장소를 골라야 하는 요구 단위로 분리합니다. 같은 domain이어도 음식군·지역·목적이 다르면 별도 Task입니다.
- 여러 장소를 나열해 추천해 달라는 말만으로 day_trip_route로 분류하지 않습니다. 일정·동선·코스·방문 순서를 요청할 때만 route intent를 사용합니다.
- 질문에 없는 조건을 추가하지 않습니다.
- mcp_only와 general_response에서는 tasks를 빈 리스트로 반환합니다.

[WeatherRequest]
- rag_mcp 또는 mcp_only일 때만 생성합니다.
- query에는 날짜·시간 표현이 보존되도록 사용자 원문 전체를 넣습니다.
- target_date와 target_time은 현재 날짜를 기준으로 해석할 수 있을 때만 채웁니다.
- location_name은 사용자가 명시한 경우에만 채웁니다.
- 위도·경도는 만들지 않습니다. 좌표는 UserContext에서 결합합니다.
- language는 사용자 질문 언어와 동일하게 설정합니다.

[Filter]
- 명시되거나 UserContext에 제공된 정보만 사용합니다.
- 현재 좌표는 filters에 복사하지 않습니다.
- 날짜는 가능한 경우 YYYY-MM-DD로 해석합니다.
- 모든 Task에 공통인 조건은 TravelQuery.filters에 넣습니다.
- 지역·날짜·시간·예산 등이 Task마다 다르면 해당 QueryTask.filters에 넣고, 전역 filters에는 억지로 하나를 고르지 않습니다.

[검증]
- rag_only: Task 있음, WeatherRequest 없음
- rag_mcp: Task 있음, WeatherRequest 있음
- mcp_only: Task 없음, WeatherRequest 있음, intent=weather_information
- general_response: Task/WeatherRequest/source_mode 없음, general_response_instruction 있음
"""

PARSER_HUMAN_PROMPT = """
[현재 날짜]
{today}

[사용자 컨텍스트]
{user_context}

[사용자 질문]
{question}

TravelQuery 스키마로 구조화하세요.
"""


def create_structured_query_chain(model_name: str = MODEL_NAME):
    llm = ChatOpenAI(model=model_name, temperature=0)
    prompt = ChatPromptTemplate.from_messages(
        [("system", PARSER_SYSTEM_PROMPT), ("human", PARSER_HUMAN_PROMPT)]
    )
    return prompt | llm.with_structured_output(TravelQuery, method="function_calling")


def create_general_answer_chain(model_name: str = MODEL_NAME):
    llm = ChatOpenAI(model=model_name, temperature=0.3)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 SeoulMate의 일반 어시스턴트입니다. 사용자 언어로 정확히 답하고, "
                "실행하지 않은 RAG나 MCP를 사용했다고 말하지 마세요.",
            ),
            ("human", "질문: {question}\n응답 지침: {instruction}"),
        ]
    )
    return prompt | llm | StrOutputParser()

## 6. Weather MCP Provider

`BackendWeatherMcpProvider`는 기존 백엔드의 실제 로컬 MCP 브리지를 재사용합니다. 도메인 팀은 날씨 코드를 다시 작성하지 않습니다.

### Weather MCP SDK 확인

Jupyter는 터미널과 다른 Python 환경을 사용할 수 있습니다. 아래 셀은 **현재 노트북 커널**에 MCP SDK가 없을 때만 설치합니다. 설치가 끝나면 같은 실행에서 계속 진행할 수 있습니다.

In [6]:
import importlib
import importlib.metadata
import importlib.util
import subprocess

if importlib.util.find_spec("mcp") is None:
    print("현재 Jupyter 커널에 MCP SDK가 없어 설치합니다:", sys.executable)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "mcp>=1.28,<2"]
    )
    importlib.invalidate_caches()

if importlib.util.find_spec("mcp") is None:
    raise RuntimeError(
        "MCP SDK 설치를 확인하지 못했습니다. 커널을 재시작한 뒤 이 셀부터 다시 실행하세요."
    )

print("MCP SDK:", importlib.metadata.version("mcp"))
print("현재 Jupyter Python:", sys.executable)

MCP SDK: 1.28.1
현재 Jupyter Python: C:\Users\user\Desktop\seoulmate\SeoulMate\backend\.venv\Scripts\python.exe


In [7]:
class BackendWeatherMcpProvider:
    async def ainvoke(self, inputs: dict[str, Any]) -> dict[str, Any]:
        try:
            from services.weather_mcp_client import get_weather_via_mcp
        except ModuleNotFoundError as exc:
            # MCP SDK 누락 때문에 LangGraph 전체가 중단되지 않도록 실패 안전 응답을 반환한다.
            return {
                "available": False,
                "is_forecast": False,
                "weather_tags": [],
                "usage_guidance": [],
                "should_affect_recommendation": False,
                "source": "weather-mcp-sdk-unavailable",
                "error": str(exc),
                "recovery": (
                    f"현재 커널({sys.executable})에서 "
                    "python -m pip install 'mcp>=1.28,<2'를 실행하세요."
                ),
            }

        return await get_weather_via_mcp(
            query=inputs["query"],
            lat=inputs["lat"],
            lng=inputs["lng"],
            language=inputs["language"],
            place_name=inputs.get("place_name"),
        )


class FixtureWeatherMcpProvider:
    # API 키와 MCP 서버 없이 그래프 분기를 검증하는 테스트 더블.

    async def ainvoke(self, inputs: dict[str, Any]) -> dict[str, Any]:
        return {
            "available": True,
            "source": "fixture-weather-mcp",
            "location": inputs.get("place_name") or "현재 위치",
            "query": inputs["query"],
            "temperature_c": 24.0,
            "precipitation_type": "none",
            "weather_tags": ["mild"],
            "usage_guidance": ["날씨는 추천의 보조 신호로만 사용합니다."],
        }

## 7. 검색 및 MCP 요청 생성 함수

In [8]:
DOMAIN_AGENT_NAMES: dict[Domain, str] = {
    Domain.CAFE: "CafeAgent",
    Domain.RESTAURANT: "RestaurantAgent",
    Domain.ACCOMMODATION: "AccommodationAgent",
    Domain.ATTRACTION: "AttractionAgent",
    Domain.ETC: "EtcAgent",
}


def compact_dict(data: dict[str, Any]) -> dict[str, Any]:
    return {
        key: value
        for key, value in data.items()
        if value is not None and value != [] and value != {}
    }


def build_metadata_filters(filters: SearchFilters, user_context: UserContext) -> dict[str, Any]:
    metadata = filters.model_dump(mode="json")
    if user_context.current_location:
        metadata["current_location"] = user_context.current_location.model_dump()
    if user_context.location_name and not metadata.get("location"):
        metadata["location"] = user_context.location_name
    return compact_dict(metadata)


def effective_task_filters(task: QueryTask, parsed_query: TravelQuery) -> SearchFilters:
    if task.filters is None:
        return parsed_query.filters
    updates = task.filters.model_dump(exclude_unset=True, exclude_none=True)
    updates = {key: value for key, value in updates.items() if value != [] and value != ""}
    return parsed_query.filters.model_copy(update=updates)


def build_search_query_text(task: QueryTask, parsed_query: TravelQuery) -> str:
    parts = [task.search_query.strip()]
    if task.themes:
        parts.append("테마: " + ", ".join(task.themes))
    if task.notes:
        parts.append("추가 조건: " + task.notes.strip())
    filters = effective_task_filters(task, parsed_query)
    if filters.location:
        parts.append("지역: " + filters.location)
    return " | ".join(parts)


def create_search_requests(
    parsed_query: TravelQuery,
    user_context: UserContext,
) -> list[SearchRequest]:
    return [
        SearchRequest(
            task_id=task.task_id,
            domain=task.domain,
            query_text=build_search_query_text(task, parsed_query),
            metadata_filters=build_metadata_filters(effective_task_filters(task, parsed_query), user_context),
            top_k=max(5, task.desired_count),
        )
        for task in parsed_query.tasks
    ]


def create_agent_requests(search_requests: list[SearchRequest]) -> list[AgentRequest]:
    return [
        AgentRequest(
            agent_name=DOMAIN_AGENT_NAMES[request.domain],
            task_id=request.task_id,
            domain=request.domain,
            search_request=request,
        )
        for request in search_requests
    ]


def create_weather_mcp_request(
    parsed_query: TravelQuery,
    user_context: UserContext,
) -> WeatherMcpRequest:
    if not parsed_query.weather_request:
        raise ValueError("날씨 경로인데 weather_request가 없습니다.")
    if not user_context.current_location:
        raise ValueError(
            "Weather MCP는 좌표가 필요합니다. 명시된 지역을 먼저 geocoding한 뒤 "
            "UserContext.current_location에 넣으세요."
        )

    place_name = (
        parsed_query.weather_request.location_name
        or parsed_query.filters.location
        or user_context.location_name
    )
    point = user_context.current_location
    return WeatherMcpRequest(
        query=parsed_query.weather_request.query,
        lat=point.latitude,
        lng=point.longitude,
        language="en" if parsed_query.language == Language.EN else "ko",
        place_name=place_name,
    )

## 8. LangGraph 노드와 세 가지 최종 GPT payload

In [9]:
def make_parse_query_node(services: GraphServices):
    async def parse_query_node(state: GraphState) -> dict[str, Any]:
        user_context = UserContext.model_validate(state.get("user_context", {}))
        parsed = await services.query_parser.ainvoke(
            {
                "today": date.today().isoformat(),
                "question": state["question"],
                "user_context": json.dumps(
                    user_context.model_dump(mode="json"), ensure_ascii=False
                ),
            }
        )
        validated = TravelQuery.model_validate(parsed)
        return {
            "parsed_query": validated,
            "source_mode": validated.source_mode.value if validated.source_mode else "general",
            "route_name": "parsed",
        }

    return parse_query_node


def decide_initial_route(state: GraphState) -> str:
    parsed = state["parsed_query"]
    if parsed.intent == Intent.GENERAL_RESPONSE:
        return "general_answer"
    if parsed.source_mode == SourceMode.MCP_ONLY:
        return "prepare_weather_mcp"
    return "generate_search_requests"


def make_general_answer_node(services: GraphServices):
    async def general_answer_node(state: GraphState) -> dict[str, Any]:
        parsed = state["parsed_query"]
        answer = await services.general_answer_generator.ainvoke(
            {
                "question": state["question"],
                "instruction": parsed.general_response_instruction,
            }
        )
        return {"final_answer": answer, "route_name": "general_llm"}

    return general_answer_node


def generate_search_requests_node(state: GraphState) -> dict[str, Any]:
    parsed = state["parsed_query"]
    context = UserContext.model_validate(state.get("user_context", {}))
    return {
        "search_requests": create_search_requests(parsed, context),
        "route_name": "search_query_generated",
    }


def decide_agent_route(state: GraphState) -> str:
    if state["parsed_query"].intent == Intent.SINGLE_PLACE_RECOMMENDATION:
        return "domain_agent_router"
    return "route_pipeline_ready"


def domain_agent_router_node(state: GraphState) -> dict[str, Any]:
    return {
        "agent_requests": create_agent_requests(state["search_requests"]),
        "route_name": "domain_agent_router",
    }


def route_pipeline_ready_node(state: GraphState) -> dict[str, Any]:
    return {
        "agent_requests": create_agent_requests(state["search_requests"]),
        "route_name": "route_pipeline_ready",
    }


def decide_after_rag_plan(state: GraphState) -> str:
    if state["parsed_query"].source_mode == SourceMode.RAG_MCP:
        return "prepare_weather_mcp"
    return "finalize_rag_only"


def prepare_weather_mcp_node(state: GraphState) -> dict[str, Any]:
    parsed = state["parsed_query"]
    context = UserContext.model_validate(state.get("user_context", {}))
    return {
        "weather_mcp_request": create_weather_mcp_request(parsed, context),
        "route_name": "weather_mcp_request_ready",
    }


def make_invoke_weather_mcp_node(services: GraphServices):
    async def invoke_weather_mcp_node(state: GraphState) -> dict[str, Any]:
        request = state["weather_mcp_request"]
        weather = await services.weather_provider.ainvoke(request.model_dump())
        return {"weather_context": weather, "route_name": "weather_mcp_called"}

    return invoke_weather_mcp_node


def decide_weather_final_route(state: GraphState) -> str:
    if state["parsed_query"].source_mode == SourceMode.MCP_ONLY:
        return "finalize_mcp_only"
    return "finalize_rag_mcp"


def finalize_rag_only_node(state: GraphState) -> dict[str, Any]:
    payload = {
        "mode": "RAG_ONLY",
        "apply_weather_reranking": False,
        "question": state["question"],
        "agent_requests": [item.model_dump(mode="json") for item in state.get("agent_requests", [])],
    }
    return {
        "final_answer_input": payload,
        "final_answer": "RAG 결과만 최종 GPT에 전달할 준비가 되었습니다.",
        "route_name": "rag_only_final",
    }


def finalize_rag_mcp_node(state: GraphState) -> dict[str, Any]:
    payload = {
        "mode": "RAG_MCP",
        "apply_weather_reranking": True,
        "question": state["question"],
        "agent_requests": [item.model_dump(mode="json") for item in state.get("agent_requests", [])],
        "weather_context": state.get("weather_context", {}),
        "next_step": "도메인 RAG 결과를 날씨로 재랭킹한 뒤 최종 GPT에 전달",
    }
    return {
        "final_answer_input": payload,
        "final_answer": "RAG와 Weather MCP를 결합할 준비가 되었습니다.",
        "route_name": "rag_mcp_final",
    }


def finalize_mcp_only_node(state: GraphState) -> dict[str, Any]:
    payload = {
        "mode": "MCP_ONLY",
        "apply_weather_reranking": False,
        "question": state["question"],
        "weather_context": state.get("weather_context", {}),
        "constraint": "장소 추천이나 RAG 근거를 추가하지 않음",
    }
    return {
        "final_answer_input": payload,
        "final_answer": "Weather MCP 결과만 최종 GPT에 전달할 준비가 되었습니다.",
        "route_name": "mcp_only_final",
    }

## 9. LangGraph 조립

In [10]:
def build_seoulmate_graph(services: GraphServices):
    builder = StateGraph(GraphState)

    builder.add_node("parse_query", make_parse_query_node(services))
    builder.add_node("general_answer", make_general_answer_node(services))
    builder.add_node("generate_search_requests", generate_search_requests_node)
    builder.add_node("domain_agent_router", domain_agent_router_node)
    builder.add_node("route_pipeline_ready", route_pipeline_ready_node)
    builder.add_node("prepare_weather_mcp", prepare_weather_mcp_node)
    builder.add_node("invoke_weather_mcp", make_invoke_weather_mcp_node(services))
    builder.add_node("finalize_rag_only", finalize_rag_only_node)
    builder.add_node("finalize_rag_mcp", finalize_rag_mcp_node)
    builder.add_node("finalize_mcp_only", finalize_mcp_only_node)

    builder.add_edge(START, "parse_query")
    builder.add_conditional_edges(
        "parse_query",
        decide_initial_route,
        {
            "general_answer": "general_answer",
            "generate_search_requests": "generate_search_requests",
            "prepare_weather_mcp": "prepare_weather_mcp",
        },
    )
    builder.add_conditional_edges(
        "generate_search_requests",
        decide_agent_route,
        {
            "domain_agent_router": "domain_agent_router",
            "route_pipeline_ready": "route_pipeline_ready",
        },
    )
    for node_name in ("domain_agent_router", "route_pipeline_ready"):
        builder.add_conditional_edges(
            node_name,
            decide_after_rag_plan,
            {
                "prepare_weather_mcp": "prepare_weather_mcp",
                "finalize_rag_only": "finalize_rag_only",
            },
        )

    builder.add_edge("prepare_weather_mcp", "invoke_weather_mcp")
    builder.add_conditional_edges(
        "invoke_weather_mcp",
        decide_weather_final_route,
        {
            "finalize_mcp_only": "finalize_mcp_only",
            "finalize_rag_mcp": "finalize_rag_mcp",
        },
    )

    for terminal_node in (
        "general_answer",
        "finalize_rag_only",
        "finalize_rag_mcp",
        "finalize_mcp_only",
    ):
        builder.add_edge(terminal_node, END)

    return builder.compile()


async def run_graph(
    graph,
    question: str,
    current_location: Optional[dict[str, float]] = None,
    location_name: Optional[str] = None,
) -> GraphState:
    context = UserContext(
        current_location=GeoPoint(**current_location) if current_location else None,
        location_name=location_name,
    )
    return await graph.ainvoke(
        {
            "question": question,
            "user_context": context.model_dump(mode="json"),
        }
    )

## 10. Fixture 기반 세 경로 테스트

In [11]:
class FixtureStructuredQueryParser:
    def __init__(self, fixtures: dict[str, TravelQuery]):
        self.fixtures = fixtures

    async def ainvoke(self, inputs: dict[str, Any]) -> TravelQuery:
        return self.fixtures[inputs["question"]]


class FixtureGeneralAnswerGenerator:
    async def ainvoke(self, inputs: dict[str, Any]) -> str:
        return f"[일반 LLM] {inputs['question']}"


Q_RAG = "조용한 감성 카페 추천해줘"
Q_BOTH = "내일 저녁에 갈 조용한 감성 카페 추천해줘"
Q_MCP = "내일 서울 날씨 알려줘"
Q_MULTI = "서울에서 2박 3일 동안 궁궐과 로컬 맛집 위주로 여행하고 싶어"
Q_GENERAL = "파이썬 리스트와 튜플 차이가 뭐야?"

fixtures = {
    Q_RAG: TravelQuery(
        language="ko",
        intent="single_place_recommendation",
        source_mode="rag_only",
        original_question=Q_RAG,
        normalized_question="서울의 조용한 감성 카페 추천",
        tasks=[QueryTask(
            task_id="task_1", domain="cafe",
            search_query="서울 조용한 감성 카페",
            themes=["조용한", "감성"], desired_count=1,
        )],
        filters=SearchFilters(location="Seoul"),
    ),
    Q_BOTH: TravelQuery(
        language="ko",
        intent="single_place_recommendation",
        source_mode="rag_mcp",
        original_question=Q_BOTH,
        normalized_question="내일 저녁 서울의 조용한 감성 카페 추천",
        tasks=[QueryTask(
            task_id="task_1", domain="cafe",
            search_query="서울 조용한 감성 카페",
            themes=["조용한", "감성"], desired_count=1,
        )],
        filters=SearchFilters(location="Seoul", time_window="19:00"),
        weather_request=WeatherRequest(
            query=Q_BOTH, location_name="Seoul",
            target_time="19:00", language="ko",
        ),
    ),
    Q_MCP: TravelQuery(
        language="ko",
        intent="weather_information",
        source_mode="mcp_only",
        original_question=Q_MCP,
        normalized_question="내일 서울 날씨",
        tasks=[],
        filters=SearchFilters(location="Seoul"),
        weather_request=WeatherRequest(
            query=Q_MCP, location_name="Seoul", language="ko",
        ),
    ),
    Q_MULTI: TravelQuery(
        language="ko",
        intent="multi_day_route",
        source_mode="rag_only",
        original_question=Q_MULTI,
        normalized_question="서울 2박 3일 궁궐과 로컬 맛집 일정",
        tasks=[
            QueryTask(task_id="task_1", slot_id="d1-palace", day_number=1, visit_date=date(2026, 7, 16), start_time="14:00", domain="attraction", search_query="서울 궁궐", desired_count=5),
            QueryTask(task_id="task_2", slot_id="d1-dinner", day_number=1, visit_date=date(2026, 7, 16), start_time="19:00", domain="restaurant", search_query="서울 로컬 맛집", desired_count=5),
        ],
        filters=SearchFilters(location="Seoul", start_date=date(2026, 7, 16), end_date=date(2026, 7, 18)),
        route_request=RouteRequest(
            destination="서울",
            period=TripPeriod(start_date=date(2026, 7, 16), end_date=date(2026, 7, 18), nights=2, days=3),
            preferred_themes=["궁궐", "로컬 맛집"],
        ),
    ),
    Q_GENERAL: TravelQuery(
        language="ko",
        intent="general_response",
        source_mode=None,
        original_question=Q_GENERAL,
        normalized_question=Q_GENERAL,
        tasks=[],
        filters=SearchFilters(),
        general_response_instruction="초보자에게 두 자료형의 차이를 설명한다.",
    ),
}

fixture_services = GraphServices(
    query_parser=FixtureStructuredQueryParser(fixtures),
    general_answer_generator=FixtureGeneralAnswerGenerator(),
    weather_provider=FixtureWeatherMcpProvider(),
)
fixture_graph = build_seoulmate_graph(fixture_services)

fixture_results = {}
for question in fixtures:
    result = await run_graph(
        fixture_graph,
        question,
        current_location={"latitude": 37.5665, "longitude": 126.9780},
        location_name="Seoul",
    )
    fixture_results[question] = result
    print("=" * 80)
    print("질문:", question)
    print("source_mode:", result.get("source_mode"))
    print("route_name:", result["route_name"])
    print("RAG 요청 수:", len(result.get("agent_requests", [])))
    print("MCP 호출:", "weather_context" in result)

질문: 조용한 감성 카페 추천해줘
source_mode: rag_only
route_name: rag_only_final
RAG 요청 수: 1
MCP 호출: False
질문: 내일 저녁에 갈 조용한 감성 카페 추천해줘
source_mode: rag_mcp
route_name: rag_mcp_final
RAG 요청 수: 1
MCP 호출: True
질문: 내일 서울 날씨 알려줘
source_mode: mcp_only
route_name: mcp_only_final
RAG 요청 수: 0
MCP 호출: True
질문: 서울에서 2박 3일 동안 궁궐과 로컬 맛집 위주로 여행하고 싶어
source_mode: rag_only
route_name: rag_only_final
RAG 요청 수: 2
MCP 호출: False
질문: 파이썬 리스트와 튜플 차이가 뭐야?
source_mode: general
route_name: general_llm
RAG 요청 수: 0
MCP 호출: False


In [12]:
assert fixture_results[Q_RAG]["route_name"] == "rag_only_final"
assert "weather_context" not in fixture_results[Q_RAG]
assert fixture_results[Q_RAG]["agent_requests"][0].agent_name == "CafeAgent"

assert fixture_results[Q_BOTH]["route_name"] == "rag_mcp_final"
assert fixture_results[Q_BOTH]["weather_context"]["available"] is True
assert len(fixture_results[Q_BOTH]["agent_requests"]) == 1

assert fixture_results[Q_MCP]["route_name"] == "mcp_only_final"
assert fixture_results[Q_MCP].get("agent_requests", []) == []
assert fixture_results[Q_MCP]["weather_context"]["available"] is True

assert fixture_results[Q_MULTI]["route_name"] == "rag_only_final"
assert len(fixture_results[Q_MULTI]["agent_requests"]) == 2

assert fixture_results[Q_GENERAL]["route_name"] == "general_llm"

print("RAG_ONLY / RAG_MCP / MCP_ONLY / general_response 분기 테스트를 모두 통과했습니다.")

RAG_ONLY / RAG_MCP / MCP_ONLY / general_response 분기 테스트를 모두 통과했습니다.


## 11. Pydantic 모순 차단 테스트

In [13]:
invalid_examples = [
    # RAG_MCP인데 날씨 요청이 없음
    {
        "language": "ko",
        "intent": "single_place_recommendation",
        "source_mode": "rag_mcp",
        "original_question": "내일 카페 추천",
        "normalized_question": "내일 카페 추천",
        "tasks": [{"task_id": "task_1", "domain": "cafe", "search_query": "카페"}],
        "filters": {},
    },
    # MCP_ONLY인데 RAG Task가 있음
    {
        "language": "ko",
        "intent": "weather_information",
        "source_mode": "mcp_only",
        "original_question": "내일 날씨",
        "normalized_question": "내일 날씨",
        "tasks": [{"task_id": "task_1", "domain": "etc", "search_query": "잘못된 검색"}],
        "filters": {},
        "weather_request": {"query": "내일 날씨", "language": "ko"},
    },
]

for index, invalid in enumerate(invalid_examples, start=1):
    try:
        TravelQuery.model_validate(invalid)
        raise AssertionError(f"잘못된 입력 {index}가 통과했습니다.")
    except ValidationError as error:
        print(f"[예상된 검증 실패 {index}]", error.errors()[0]["msg"])

[예상된 검증 실패 1] Value error, RAG_MCP에서는 weather_request가 필요합니다.
[예상된 검증 실패 2] Value error, MCP_ONLY에서는 도메인 Task가 없어야 합니다.


## 12. 실제 OpenAI Parser + 실제 Weather MCP 테스트

1. 백엔드 `.env`에 `OPENAI_API_KEY`, `KMA_API_KEY`를 설정합니다.
2. 별도 터미널에서 `weather_mcp_server.py`를 실행합니다.
3. 아래 셀을 실행합니다.

실제 Domain RAG는 아직 이 노트북 범위 밖이므로 `AgentRequest`까지만 생성됩니다. 음식점·카페·숙박·문화시설 Agent가 검색 결과를 반환하면 `final_answer_input`에 후보를 추가하고 각 모드의 최종 GPT 체인으로 넘기면 됩니다.

In [14]:
RUN_LIVE_PARSER_MCP = False

if RUN_LIVE_PARSER_MCP and HAS_OPENAI_KEY:
    live_services = GraphServices(
        query_parser=create_structured_query_chain(),
        general_answer_generator=create_general_answer_chain(),
        weather_provider=BackendWeatherMcpProvider(),
    )
    live_graph = build_seoulmate_graph(live_services)

    live_questions = [
        "조용한 감성 카페 추천해줘",
        "내일 저녁에 갈 조용한 감성 카페 추천해줘",
        "내일 서울 날씨 알려줘",
    ]

    for question in live_questions:
        result = await run_graph(
            live_graph,
            question,
            current_location={"latitude": 37.5665, "longitude": 126.9780},
            location_name="Seoul",
        )
        print("=" * 80)
        print("질문:", question)
        print("Structured Query:")
        print(result["parsed_query"].model_dump_json(indent=2))
        print("최종 경로:", result["route_name"])
        print("최종 GPT 입력:")
        print(json.dumps(result.get("final_answer_input", {}), ensure_ascii=False, indent=2))
else:
    print("실제 LLM/MCP 테스트를 건너뜁니다. 실행하려면 RUN_LIVE_PARSER_MCP=True로 변경하세요.")

실제 LLM/MCP 테스트를 건너뜁니다. 실행하려면 RUN_LIVE_PARSER_MCP=True로 변경하세요.


## 13. 도메인 팀 연결 지점

각 팀은 날씨 로직을 다시 만들지 않고 다음 두 부분만 구현합니다.

1. 자신의 `AgentRequest`를 받아 VectorDB/RDB 검색 결과를 반환하는 Domain Agent
2. 후보와 `weather_context`를 받아 보조 점수를 적용하는 도메인별 재랭커

최종 GPT 입력 규칙:

- `RAG_ONLY`: 도메인 검색 결과만 전달하고 날씨 재랭킹을 호출하지 않음
- `RAG_MCP`: 도메인 검색 결과 + Weather MCP 결과에만 날씨 재랭킹 적용
- `MCP_ONLY`: Weather MCP 결과만 전달하고 장소를 만들어내지 않음

주의: 사용자가 현재 위치가 아닌 다른 지역을 명시하면 그 지역을 좌표로 변환하는 geocoding 노드를 `prepare_weather_mcp` 앞에 추가해야 합니다. LLM이 좌표를 추측하게 하면 안 됩니다.